In [1]:
import pandas as pd

df = pd.read_csv("data/global_excel_resampled_scaled.csv", sep=";")

In [3]:
print(df.columns)

Index(['Unnamed: 0', 'original_shape_Elongation', 'original_shape_Flatness',
       'original_shape_LeastAxisLength', 'original_shape_MajorAxisLength',
       'original_shape_Maximum2DDiameterColumn',
       'original_shape_Maximum2DDiameterRow',
       'original_shape_Maximum2DDiameterSlice',
       'original_shape_Maximum3DDiameter', 'original_shape_MeshVolume',
       ...
       'original_glcm_JointEntropy', 'original_glcm_MCC',
       'original_glcm_MaximumProbability', 'original_glcm_SumAverage',
       'original_glcm_SumEntropy', 'original_glcm_SumSquares', 'classe_name',
       'patient_num', 'temps_inj', 'id'],
      dtype='object', length=111)


In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. FILTRAGE INITIAL ET SÉLECTION DES PHASES
# ==========================================

# Définition des phases requises et de leur ordre strict
phases_cibles = ["PORT", "ART", "TARD"]

# On ne garde temporairement que les lignes qui concernent ces 3 phases
df = df[df["temps_inj"].isin(phases_cibles)]

# ==========================================
# 2. IDENTIFICATION DES PATIENTS ÉLIGIBLES
# ==========================================

# Un patient est valide s'il possède exactement les 3 phases distinctes
phases_par_patient = df.groupby("id")["temps_inj"].nunique()
patients_valides = phases_par_patient[phases_par_patient == 3].index

# Filtrage du DataFrame pour ne garder que ces patients complets
df = df[df["id"].isin(patients_valides)]

# Transforme la colonne en type Categorical pour imposer l'ordre de tri des colonnes plus tard
df["temps_inj"] = pd.Categorical(
    df["temps_inj"], categories=phases_cibles, ordered=True
)

# ==========================================
# 3. PIVOTEMENT ET RESTRUCTURATION EN WIDE
# ==========================================

# Identification des colonnes de métadonnées à garder comme index de ligne
metadata_cols = ["id", "patient_num", "classe_name"]

# Identification automatique de toutes les colonnes de features radiomiques
feature_cols = [
    col
    for col in df.columns
    if col not in metadata_cols + ["temps_inj", "Unnamed: 0"]
]

# Pivotement du DataFrame
df_wide = df.pivot(
    index=metadata_cols, columns="temps_inj", values=feature_cols
)

# Réorganisation des niveaux d'index des colonnes : on bascule la phase en premier niveau
df_wide = df_wide.swaplevel(0, 1, axis=1)

# Tri des colonnes par le niveau 0 (les phases), garantissant l'ordre : PORT -> ART -> TARD
# sort_remaining=False permet de conserver l'ordre d'origine des features radiomiques
df_wide = df_wide.sort_index(axis=1, level=0, sort_remaining=False)

# ==========================================
# 4. APLATISSEMENT ET NETTOYAGE FINAL
# ==========================================

# Reconstruction des noms de colonnes sous la forme : nom_feature_PHASE
df_wide.columns = [f"{feat}_{phase}" for phase, feat in df_wide.columns]

# Réinitialisation de l'index pour repasser 'id', 'patient_num' et 'classe_name' en colonnes normales
df_final = df_wide.reset_index()

In [8]:
df["id"] = df["id"].astype(int)
df_final.to_csv("./data/flattened_global_scaled.csv", sep=";", index=False, encoding="utf-8-sig")